# 🤖 AutoDoc AI

> *Paste your code of any language, a frontier language model reads it, understands it, and writes meaningful inline comments for you in seconds.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="https://media2.giphy.com/media/v1.Y2lkPTc5MGI3NjExMXR1c256cDZzNjFoZGd2M3Nkampra2xzbHM3MXY4dnI2ejdsdGh6eSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/HwmdjTpOCR4fuhxN54/giphy.gif" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">💡 Customize Your Comment Style!</h2>
            <span style="color:#f71;">You can control <strong>how</strong> the AI comments your code by editing the prompt in the code cell below. Try instructions like:<br><br>
            &bull; <code>"Add short, one-line comments only."</code><br>
            &bull; <code>"Write detailed, beginner-friendly explanations."</code><br>
            &bull; <code>"Add docstrings only, skip inline comments."</code><br>
            &bull; <code>"Comment in the style of a senior engineer."</code><br><br>
            Just modify the <code>comment_style</code> variable before running!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="https://media3.giphy.com/media/v1.Y2lkPTc5MGI3NjExcmwwNmI5a2F2MXJtc2piODl6ZzlpOXVjYmpxbXA1d3lodXVheWZveSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LdOyjZ7io5Msw/giphy.gif" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, we use closed-source models via API — specifically GPT (OpenAI) and Gemini (Google). Make sure you have valid API keys for both set up in your <code>.env</code> file before running.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import re

In [ ]:
# Load environment variables from .env file, overriding any existing system variables

load_dotenv(override=True)

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Validate OpenAI API key and print first 8 characters as confirmation
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# Validate Google API key (optional) and print first 2 characters as confirmation
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:6]}")
else:
    print("Google API Key not set (and this is optional)")

In [ ]:
# Initialize OpenAI client using the default API key from environment

openai = OpenAI()

# Define the Gemini API base URL using Google's OpenAI-compatible endpoint
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

# Initialize Gemini client using Google API key and custom base URL
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
# List of cheap models from OpenAI and Google Gemini
models = ["gpt-5-nano", "gpt-5-mini", "gpt-4o-mini", "gpt-4o", "gemini-2.5-flash", "gemini-2.5-flash-lite"]

# Map model prefixes to their corresponding API clients
clients = {
    "gpt": openai,
    "gemini": gemini
}

def get_client(model):
    # Return the OpenAI client for GPT models
    if model.startswith("gpt"):
        return clients["gpt"]
    # Return the Gemini client for Gemini models
    elif model.startswith("gemini"):
        return clients["gemini"]
    # Raise an error if the model prefix is unrecognized
    else:
        raise ValueError(f"Unknown model: {model}")

## And now, on with the main task

In [ ]:
# System prompt instructing the AI to add comments to code
system_prompt = """
Your task is to add comments to the provided code.
Respond only with the commented code. Do not provide any explanation outside of the comments.
The comments should be clear, accurate, and match the style specified by the user.
"""

# Build a user prompt that includes the code and comment style preference
def user_prompt_for(code, language, comment_style):
    return f"""
Add {comment_style} comments to the following {language} code.
Respond only with the commented code, no extra explanation.

Code to comment:

```{language}
{code}
```
"""

In [ ]:
# Build the messages list with system and user roles for the chat completion API
def messages_for(code, language, comment_style):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(code, language, comment_style)}
    ]

In [ ]:
# Add comments to the provided code using the specified model
def comment_code(model, code, language, comment_style):
    # Select the appropriate API client based on the model name
    client = get_client(model)
    
    # Send the conversion request to the model
    response = client.chat.completions.create(model=model, messages=messages_for(code, language, comment_style))
    reply = response.choices[0].message.content
    
    # Strip markdown code fences from the response if present
    reply = re.sub(r'```[\w]*\n?', '', reply).replace('```', '').strip()
    return reply

In [ ]:
from styles import CSS

with gr.Blocks(title="🤖 AutoDoc AI") as ui:
    gr.Markdown("## 🤖 AutoDoc AI")

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            code_input = gr.Code(
                label="Your Code",
                value="",
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            code_output = gr.Code(
                label="Commented Code",
                value="",
                language="python",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0], show_label=False)
        language = gr.Dropdown(
            ["python", "javascript", "java", "c", "cpp", "typescript", "go", "rust", "php", "ruby"],
            value="python",
            show_label=False
        )
        comment_style = gr.Dropdown(
            ["Short inline", "Detailed", "Beginner-friendly", "Docstrings only", "Senior engineer style"],
            value="Short inline",
            show_label=False
        )
        generate = gr.Button("Add Comments", elem_classes=["convert-btn"])

    # When language changes, update syntax highlighting on both boxes
    language.change(
        fn=lambda lang: (gr.Code(language=lang), gr.Code(language=lang)),
        inputs=[language],
        outputs=[code_input, code_output]
    )

    generate.click(fn=comment_code, inputs=[model, code_input, language, comment_style], outputs=[code_output])

ui.launch(inbrowser=True, css=CSS, theme=gr.themes.Monochrome())